In [ ]:
# @title Step 1: get the list of URLs with market data the current END_YEAR =2026 (the getting market data part is time consuming and takes +1hour )
import re
import requests
from bs4 import BeautifulSoup
import urllib.parse

# Years to scan
START_YEAR = 2009
END_YEAR   = 2026          # inclusive

def ba2_links_for_year(year):
    """Return all .ba2 URLs found on the market‑history page for a given year."""
    page_url = f"https://data.everef.net/market-history/{year}/"
    resp = requests.get(page_url)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    links = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        full_url = requests.compat.urljoin(page_url, href)

        if re.search(r"\.bz2$", full_url, re.IGNORECASE):
            links.append(full_url)

    return links

def is_valid_bz2_url(url: str) -> bool:
    """
    Return True if `url` looks like a proper HTTP(S) URL and ends with '.bz2'.
    """
    try:
        parsed = urllib.parse.urlparse(url)
        # Must have scheme and network location
        if parsed.scheme not in {"http", "https"} or not parsed.netloc:
            return False
        # Path must end with .bz2 (case‑insensitive)
        return parsed.path.lower().endswith(".bz2")
    except Exception:
        return False

def collect_all_ba2_links(start=START_YEAR, end=END_YEAR):
    """Gather .ba2 URLs from every year in the range."""
    all_links = []
    for yr in range(start, end + 1):
        try:
            yearly = ba2_links_for_year(yr)
            print(f"Year {yr}: {len(yearly)} .ba2 files found")
            all_links.extend(yearly)
        except requests.HTTPError as e:
            # Some years may be missing; just report and continue
            print(f"Year {yr}: error – {e}")
    return all_links

if __name__ == "__main__":
    ba2_urls = collect_all_ba2_links()
    # Example: write the list to a text file
    with open("ba2_urls.txt", "w") as f:
        for u in ba2_urls:
            if is_valid_bz2_url(u):
                f.write(u + "\n")
    print(f"\nTotal .ba2 URLs collected: {len(ba2_urls)}")


Year 2009: 365 .ba2 files found
Year 2010: 365 .ba2 files found
Year 2011: 365 .ba2 files found
Year 2012: 366 .ba2 files found
Year 2013: 365 .ba2 files found
Year 2014: 365 .ba2 files found
Year 2015: 365 .ba2 files found
Year 2016: 366 .ba2 files found
Year 2017: 365 .ba2 files found
Year 2018: 365 .ba2 files found
Year 2019: 365 .ba2 files found
Year 2020: 366 .ba2 files found
Year 2021: 365 .ba2 files found
Year 2022: 365 .ba2 files found
Year 2023: 365 .ba2 files found
Year 2024: 366 .ba2 files found
Year 2025: 365 .ba2 files found
Year 2026: 123 .ba2 files found

Total .ba2 URLs collected: 6332


In [ ]:
# @title Step 2: Get the TYPENAMES for each TYPEID (as the downloaded market data has only typeID and not names, which we need to browse )
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Download the EVE Online static‑data zip, extract *types.yaml*,
pull out the fields `typeID` and the English `typeName`,
and write them to an Excel file (TYPEID | TYPENAME).

Requires:
    requests, pyyaml, pandas, openpyxl
Install with:
    pip install requests pyyaml pandas openpyxl
"""

import io
import zipfile
import requests
import yaml
import pandas as pd

# ----------------------------------------------------------------------
# 1️⃣  Download the zip file
# ----------------------------------------------------------------------
ZIP_URL = "https://data.everef.net/ccp/sde/eve-online-static-data-latest-yaml.zip"
print("Downloading zip …")
resp = requests.get(ZIP_URL, timeout=30)
resp.raise_for_status()               # will raise if download failed

# ----------------------------------------------------------------------
# 2️⃣  Open the zip in memory and read *types.yaml*
# ----------------------------------------------------------------------
with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
    # the file is stored under the path `sde/fsd/types.yaml`
    yaml_name = "types.yaml"
    if yaml_name not in zf.namelist():
        raise FileNotFoundError(f"{yaml_name} not found inside the zip")
    print(f"Extracting {yaml_name} …")
    with zf.open(yaml_name) as yaml_file:
        raw_yaml = yaml_file.read().decode("utf-8")

# ----------------------------------------------------------------------
# 3️⃣  Parse the YAML – it is a dict keyed by the numeric typeID
# ----------------------------------------------------------------------
print("Parsing YAML …")
data = yaml.safe_load(raw_yaml)   # `data` => {199: {...}, 200: {...}, …}

# ----------------------------------------------------------------------
# 4️⃣  Build a list of rows with the required fields
# ----------------------------------------------------------------------
rows = []
for type_id, info in data.items():
    # `name` is a mapping of locale → string; we want the English entry
    en_name = info.get("name", {}).get("en", "")
    rows.append({"TYPEID": int(type_id), "TYPENAME": en_name})

# ----------------------------------------------------------------------
# 5️⃣  Convert to a DataFrame and write the Excel file
# ----------------------------------------------------------------------
df = pd.DataFrame(rows)
excel_path = "eve_types.xls"
df.to_excel(excel_path, index=False, engine="openpyxl")
print(f"✔️  Excel file written to {excel_path}")


Extracting types.yaml …
Parsing YAML …
✔️  Excel file written to eve_types.xls


In [ ]:
# @title Step 3: Create the .csv file from which the graphs are to be created ( this is where the market data is downloaded)
# --------------------------------------------------------------
# Imports & helpers
# --------------------------------------------------------------
import os
import csv
import io
import bz2
import requests
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

def sniff_delimiter(sample: bytes) -> str:
    """Return the most likely delimiter for a CSV snippet."""
    try:
        dialect = csv.Sniffer().sniff(sample.decode('utf-8', errors='ignore'))
        return dialect.delimiter
    except csv.Error:
        return "\t"      # fallback

# --------------------------------------------------------------
# Core worker – processes a single URL
# --------------------------------------------------------------
def process_url(url: str, output_path: Path, inv_df: pd.DataFrame) -> str:
    try:
        # ── download / decompress (unchanged) ────────────────────────
        resp = requests.get(url, stream=True, timeout=30)
        resp.raise_for_status()
        tmp_path = Path("/tmp") / Path(url).name
        with tmp_path.open("wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)

        with bz2.open(tmp_path, "rb") as f_in:
            raw_bytes = f_in.read()

        # ── detect delimiter & read CSV ───────────────────────────────
        delimiter = sniff_delimiter(raw_bytes[:5_000])
        try:
            df = pd.read_csv(
                io.BytesIO(raw_bytes),
                sep=delimiter,
                dtype=str,          # keep everything as string initially
                low_memory=False,
            )
        except pd.errors.ParserError:
            df = pd.read_csv(
                io.BytesIO(raw_bytes),
                sep=delimiter,
                header=None,
                names=["date", "region_id", "type_id", "average",
                       "highest", "lowest", "volume", "order_count"],
                dtype=str,
                low_memory=False,
            )

        # ── normalise column names ───────────────────────────────────
        df.columns = [c.strip().lower() for c in df.columns]
        if "region_id" not in df.columns:
            raise KeyError(f"region_id column missing – {df.columns}")

        # ── cast numeric columns ─────────────────────────────────────
        df["region_id"] = pd.to_numeric(df["region_id"], errors="coerce")
        # **IMPORTANT** – make type_id numeric so it matches inv_df
        df["type_id"] = pd.to_numeric(df["type_id"], errors="coerce")

        # ── filter wanted region_ids ─────────────────────────────────
        wanted_ids = [10000002, 19000001]
        filtered = df[df["region_id"].isin(wanted_ids)].copy()

        # ── date handling ─────────────────────────────────────────────
        filtered["date"] = pd.to_datetime(filtered["date"], utc=True).dt.strftime("%Y-%m-%d")

        # ── price_in_plex calculation ─────────────────────────────────
        filtered["average"] = pd.to_numeric(filtered["average"], errors="coerce")
        plex_price = filtered.loc[filtered["type_id"] == 44992, "average"].iloc[0]
        filtered["price_in_plex"] = filtered["average"] / plex_price

        # ── attach type_name (now both sides are int64) ─────────────────
        filtered = filtered.merge(inv_df, on="type_id", how="left")

        # ── write to final CSV ───────────────────────────────────────
        if not filtered.empty:
            out_cols = ["date", "average", "volume", "price_in_plex", "type_name"]
            filtered.to_csv(
                output_path,
                mode="a",
                header=False,
                index=False,
                columns=out_cols,
            )

        tmp_path.unlink(missing_ok=True)
        return f"✅ {url} – {len(filtered)} rows added"

    except Exception as exc:
        return f"❌ {url} – {exc}"

# --------------------------------------------------------------
# Setup (run once)
# --------------------------------------------------------------
# Paths
url_file   = Path("/content/ba2_urls.txt")               # one URL per line
output_csv = Path("/content/Book1.csv")
inv_path   = Path("/content/eve_types.xls")        # Excel lookup table

# ---- Load the Excel lookup table only once ----
inv_df = pd.read_excel(
    inv_path,
    usecols=["TYPEID", "TYPENAME"],
    dtype={"TYPEID": "int64", "TYPENAME": "string"},
)
inv_df = inv_df.rename(columns={"TYPEID": "type_id", "TYPENAME": "type_name"})

# ---- Create the output file with a header if it doesn’t exist ----
if not output_csv.exists():
    header_cols = ["date", "average", "volume", "price_in_plex", "type_name"]
    pd.DataFrame(columns=header_cols).to_csv(output_csv, index=False)

# ---- Read URLs to process ----
with url_file.open("r") as f:
    urls = [line.strip() for line in f if line.strip()]

# --------------------------------------------------------------
# Parallel execution
# --------------------------------------------------------------
max_workers = 30   # adapt to the VM / network capacity
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {
        executor.submit(process_url, url, output_csv, inv_df): url
        for url in urls
    }

    for fut in as_completed(futures):
        print(fut.result())


Streaming output truncated to the last 5000 lines.
✅ https://data.everef.net/market-history/2012/market-history-2012-08-18.csv.bz2 – 5909 rows added
✅ https://data.everef.net/market-history/2012/market-history-2012-09-04.csv.bz2 – 5693 rows added
✅ https://data.everef.net/market-history/2012/market-history-2012-09-08.csv.bz2 – 5758 rows added
✅ https://data.everef.net/market-history/2012/market-history-2012-08-28.csv.bz2 – 5698 rows added
✅ https://data.everef.net/market-history/2012/market-history-2012-09-10.csv.bz2 – 5686 rows added
✅ https://data.everef.net/market-history/2012/market-history-2012-08-21.csv.bz2 – 5786 rows added
✅ https://data.everef.net/market-history/2012/market-history-2012-09-03.csv.bz2 – 5722 rows added
✅ https://data.everef.net/market-history/2012/market-history-2012-09-11.csv.bz2 – 5759 rows added
✅ https://data.everef.net/market-history/2012/market-history-2012-08-24.csv.bz2 – 5779 rows added
✅ https://data.everef.net/market-history/2012/market-history-2012-0

In [ ]:
# @title Step 4: Use the .csv file (named Book1.csv) to create the tool, one can access the tool from the public URL created so long as the colab instance is running
# -------------------------------------------------
# 1️⃣ Imports & selective CSV load
# -------------------------------------------------
import pandas as pd
import gradio as gr
import plotly.graph_objects as go

DATA_PATH = "/content/Book1.csv"
USE_COLS = ["date", "average", "volume", "price_in_plex", "type_name"]

df = pd.read_csv(DATA_PATH, usecols=USE_COLS, parse_dates=["date"])
numeric_cols = ["average", "volume", "price_in_plex"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

type_options = sorted(df["type_name"].dropna().unique().tolist())

# -------------------------------------------------
# 2️⃣ Figure builder (price =line, volume = bar)
# -------------------------------------------------
def build_chart(selected_types, y_axes, dark_mode, *metric_colours):
    """
    selected_types : list of type_name strings
    y_axes         : list of metric column names (subset of numeric_cols)
    dark_mode      : bool
    *metric_colours: colour for each metric in numeric_cols (fixed order)
    """
    if not selected_types:
        return gr.Plot.update(value=None, label="Select at least one type_name")
    if not y_axes:
        return gr.Plot.update(value=None, label="Select at least one metric")

    # colour → metric (fixed order)
    colour_map = {metric: metric_colours[i] for i, metric in enumerate(numeric_cols) if metric in y_axes}

    # filter data for the chosen types
    sub = df[df["type_name"].isin(selected_types)]

    # one row per date‑type (first non‑null for each metric)
    agg_dict = {col: "first" for col in y_axes}
    sub_one = sub.groupby(["date", "type_name"], as_index=False).agg(agg_dict)

    fig = go.Figure()

    for t in selected_types:
        df_t = sub_one[sub_one["type_name"] == t]

        # ---- price line (if requested) ----
        if "price_in_plex" in y_axes:
            fig.add_trace(
                go.Scatter(
                    x=df_t["date"],
                    y=df_t["price_in_plex"],
                    name=f"{t} – price",
                    mode="lines",
                    line=dict(color=colour_map["price_in_plex"]),
                    yaxis="y1",
                )
            )

        # ---- volume bar (if requested) ----
        if "volume" in y_axes:
            fig.add_trace(
                go.Bar(
                    x=df_t["date"],
                    y=df_t["volume"],
                    name=f"{t} – volume",
                    marker=dict(color=colour_map["volume"]),
                    yaxis="y2",
                    opacity=0.6,
                )
            )

    # ---- layout with two y‑axes ----
    template = "plotly_dark" if dark_mode else "plotly_white"
    fig.update_layout(
        template=template,
        title="Price (line) & Volume (bars) per type_name",
        xaxis_title="Date",
        yaxis=dict(
            title="Price (in plex)",
            side="left",
            showgrid=False,
        ),
        yaxis2=dict(
            title="Volume",
            overlaying="y",
            side="right",
            showgrid=False,
        ),
        legend_title="Series",
        hovermode="x unified",
        bargap=0.2,
        margin=dict(l=60, r=60, t=60, b=40),
    )
    return fig

# -------------------------------------------------
# 3️⃣ Gradio UI (unchanged apart from the plot label)
# -------------------------------------------------
with gr.Blocks() as demo:
    gr.Markdown("# 📈 Inflation → Price & Volume")

    with gr.Row():
        type_dropdown = gr.Dropdown(
            choices=type_options,
            label="type_name (select one or more)",
            multiselect=True,
        )
        y_selector = gr.CheckboxGroup(
            choices=numeric_cols,
            label="Metrics to plot (price‑line / volume‑bar)",
            value=["price_in_plex"],   # default to price line
        )
        dark_toggle = gr.Checkbox(label="Dark mode", value=False)

    metric_colour_pickers = []
    with gr.Row():
        for metric in numeric_cols:
            cp = gr.ColorPicker(
                label=f"Colour for **{metric}**",
                value="#1f77b4",
            )
            metric_colour_pickers.append(cp)

    plot = gr.Plot(label="Price‑line & Volume‑bar")
    update_btn = gr.Button("Update chart")

    update_btn.click(
        fn=build_chart,
        inputs=[type_dropdown, y_selector, dark_toggle, *metric_colour_pickers],
        outputs=plot,
    )

if __name__ == "__main__":
    # In Colab run: demo.launch(share=True)
    demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4ecc2cef46e23ebd1a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
